# SafeCheck AI — Sistem Deteksi Penipuan Digital

**Mata Kuliah:** Artificial Intelligence  
**Nama:** Maria Devi Aparilia  
**NIM:** 24110400032  

---

## Tentang Proyek

SafeCheck AI adalah sistem RAG (Retrieval-Augmented Generation) yang membantu pengguna mendeteksi pesan penipuan digital (phishing, scam, social engineering).

**Sumber Dokumen:**
- BSSN: Lanskap Keamanan Siber Indonesia 2024
- OJK: Panduan Strategi Anti Fraud (ITSK) 2024  
- Indonesia SMS Spam Dataset (GitHub: bopbi)

**RAG Pipeline:**
```
User Query → Embedding → Vector Search → Retrieved Context → LLM → Jawaban + Risk Level
```

---

## ⚙️ Sel 1 — Instalasi Library

In [ ]:
# Install semua library yang dibutuhkan
!pip install -q anthropic sentence-transformers faiss-cpu pandas numpy

## 🔑 Sel 2 — Konfigurasi API Key

> **Cara mendapatkan API Key:** Daftar di https://console.anthropic.com → API Keys → Create Key

In [ ]:
import os
from getpass import getpass

# Masukkan Anthropic API Key kamu di sini
# Cara aman: gunakan Colab Secrets (ikon kunci di sidebar kiri)
try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print("✅ API Key berhasil dimuat dari Colab Secrets")
except:
    ANTHROPIC_API_KEY = getpass("🔑 Masukkan Anthropic API Key kamu: ")

os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

## 📚 Sel 3 — Knowledge Base (Dokumen RAG)

Berikut adalah representasi chunked dari 3 sumber dokumen utama.  
Setiap chunk sudah diberi metadata: `source`, `category`, `modus_type`, `risk_level`, `chunk_id`.

In [ ]:
# ============================================================
# KNOWLEDGE BASE — Representasi Chunked dari 3 Dokumen Sumber
# ============================================================
# Dalam implementasi produksi: load dari PDF/CSV asli lalu chunk otomatis
# Di sini disimulasikan manual sesuai rencana chunking di dokumen UTS

KNOWLEDGE_BASE = [
    # ── CHUNK A: Phishing Email ──────────────────────────────────────
    {
        "chunk_id": "chunk_A_001",
        "category": "phishing_email",
        "modus_type": "phishing",
        "risk_level": "tinggi",
        "source": "BSSN Lanskap Keamanan Siber Indonesia 2024",
        "source_url": "https://alika.pesisirbaratkab.go.id/ebook/lanskap-keamanan-siber-indonesia-2024",
        "document_type": "laporan_resmi",
        "author": "BSSN",
        "language": "Indonesian",
        "content": (
            "Phishing email perbankan adalah modus di mana pelaku membuat email tampak resmi "
            "menggunakan teknik typosquatting domain, contohnya bca-secure.com alih-alih bca.co.id. "
            "Email berisi ancaman pemblokiran akun atau klaim verifikasi mendesak dengan link ke "
            "halaman login palsu. Ciri-ciri: domain pengirim tidak resmi, ada urgensi waktu, "
            "link mengarah ke situs tiruan. Selalu periksa domain pengirim dan jangan klik link "
            "dari email yang tidak diminta."
        )
    },
    # ── CHUNK B: Scam Hadiah ─────────────────────────────────────────
    {
        "chunk_id": "chunk_B_001",
        "category": "scam_hadiah",
        "modus_type": "scam_hadiah",
        "risk_level": "tinggi",
        "source": "BSSN Lanskap Keamanan Siber Indonesia 2024",
        "source_url": "https://alika.pesisirbaratkab.go.id/ebook/lanskap-keamanan-siber-indonesia-2024",
        "document_type": "laporan_resmi",
        "author": "BSSN",
        "language": "Indonesian",
        "content": (
            "Modus scam hadiah SMS dan WhatsApp: pelaku mengirim pesan mengatasnamakan bank atau "
            "instansi resmi seperti BRI, BCA, atau Telkomsel, mengklaim penerima memenangkan "
            "hadiah besar. Indikator bahaya: penggunaan URL shortener seperti bit.ly atau tinyurl, "
            "penciptaan urgensi waktu ('klaim dalam 24 jam sebelum hangus'), permintaan transfer "
            "biaya administrasi atau pajak hadiah, link mengarah ke situs phishing. "
            "Pengguna diminta mengisi data pribadi atau melakukan transfer terlebih dahulu."
        )
    },
    {
        "chunk_id": "chunk_B_002",
        "category": "scam_hadiah",
        "modus_type": "scam_bea_cukai",
        "risk_level": "tinggi",
        "source": "BSSN Lanskap Keamanan Siber Indonesia 2024",
        "source_url": "https://alika.pesisirbaratkab.go.id/ebook/lanskap-keamanan-siber-indonesia-2024",
        "document_type": "laporan_resmi",
        "author": "BSSN",
        "language": "Indonesian",
        "content": (
            "Modus bea cukai palsu dan paket tertahan: pelaku mengirim SMS atau WhatsApp mengaku "
            "sebagai petugas Pos Indonesia, JNE, atau J&T, mengklaim ada paket tertahan di gudang "
            "bea cukai dan meminta pembayaran biaya pencairan. Ini adalah penipuan karena jasa "
            "pengiriman resmi tidak pernah meminta pembayaran melalui transfer ke rekening pribadi. "
            "Verifikasi hanya melalui website resmi atau call center resmi jasa pengiriman tersebut."
        )
    },
    # ── CHUNK C: Social Engineering ──────────────────────────────────
    {
        "chunk_id": "chunk_C_001",
        "category": "social_engineering",
        "modus_type": "impersonasi_cs",
        "risk_level": "tinggi",
        "source": "OJK Panduan Strategi Anti Fraud (ITSK) 2024",
        "source_url": "https://ojk.go.id/id/Publikasi/Roadmap-dan-Pedoman/ITSK/Documents/PANDUAN%20STRATEGI%20ANTI%20FRAUD%20(ITSK)%202024.pdf",
        "document_type": "panduan",
        "author": "OJK",
        "language": "Indonesian",
        "content": (
            "Impersonasi customer service bank atau e-commerce: pelaku menghubungi korban melalui "
            "telepon atau WhatsApp dan mengaku sebagai CS resmi dari bank, Tokopedia, Shopee, atau "
            "platform lain. Lembaga keuangan TIDAK PERNAH meminta OTP, PIN, kata sandi, atau kode "
            "CVV melalui telepon dalam kondisi apapun. Nomor yang digunakan bukan dari aplikasi "
            "resmi adalah tanda bahaya. Jika menerima telepon seperti ini, segera tutup dan "
            "verifikasi langsung ke kanal resmi. Laporkan ke OJK di nomor 157 atau layanan "
            "pengaduan resmi bank terkait."
        )
    },
    {
        "chunk_id": "chunk_C_002",
        "category": "social_engineering",
        "modus_type": "ancaman_blokir",
        "risk_level": "tinggi",
        "source": "OJK Panduan Strategi Anti Fraud (ITSK) 2024",
        "source_url": "https://ojk.go.id/id/Publikasi/Roadmap-dan-Pedoman/ITSK/Documents/PANDUAN%20STRATEGI%20ANTI%20FRAUD%20(ITSK)%202024.pdf",
        "document_type": "panduan",
        "author": "OJK",
        "language": "Indonesian",
        "content": (
            "Modus ancaman pemblokiran akun: pelaku mengirim pesan mengklaim akun perbankan atau "
            "e-commerce pengguna akan diblokir jika tidak segera melakukan verifikasi melalui link "
            "atau nomor tertentu. Ini adalah teknik social engineering berbasis rasa takut (fear-based "
            "manipulation). Bank resmi dan platform e-commerce tidak pernah meminta verifikasi "
            "melalui link di SMS atau WhatsApp. Proses resmi selalu melalui aplikasi resmi yang "
            "sudah terinstall atau website dengan domain resmi yang terverifikasi."
        )
    },
    # ── CHUNK D: Data Harvesting ──────────────────────────────────────
    {
        "chunk_id": "chunk_D_001",
        "category": "data_harvesting",
        "modus_type": "lowongan_palsu",
        "risk_level": "tinggi",
        "source": "BSSN Lanskap Keamanan Siber Indonesia 2024",
        "source_url": "https://alika.pesisirbaratkab.go.id/ebook/lanskap-keamanan-siber-indonesia-2024",
        "document_type": "laporan_resmi",
        "author": "BSSN",
        "language": "Indonesian",
        "content": (
            "Penipuan lowongan kerja sebagai modus pengumpulan data pribadi: pelaku menawarkan "
            "lowongan dengan gaji tidak realistis dan meminta dokumen sensitif sebelum proses "
            "interview resmi. Tanda bahaya: permintaan foto KTP, selfie dengan KTP, nomor rekening, "
            "atau scan dokumen identitas sebelum ada kontrak kerja. Perusahaan legitim tidak pernah "
            "meminta dokumen sensitif di awal proses rekrutmen. Data yang dikumpulkan digunakan "
            "untuk pembukaan rekening ilegal, pinjaman online tanpa sepengetahuan korban, atau "
            "penipuan finansial lainnya."
        )
    },
    # ── CHUNK E: Contoh SMS Spam Nyata ────────────────────────────────
    {
        "chunk_id": "chunk_E_001",
        "category": "sms_spam_example",
        "modus_type": "scam_hadiah",
        "risk_level": "tinggi",
        "source": "Indonesia SMS Spam Dataset",
        "source_url": "https://github.com/bopbi/indonesia-sms-spam-dataset",
        "document_type": "dataset_pesan",
        "author": "bopbi",
        "language": "Indonesian",
        "content": (
            "Contoh SMS spam nyata berlabel berbahaya dari dataset Indonesia: "
            "'Selamat! Nomor Anda terpilih memenangkan hadiah mobil. Hubungi segera sebelum hangus.' "
            "'PENGUMUMAN RESMI: Anda mendapat hadiah undian Rp50 juta. Klaim sebelum 24 jam.' "
            "'Paket Anda tertahan di bea cukai, bayar Rp150.000 untuk pencairan ke rekening ini.' "
            "'Akun Anda akan diblokir permanen, verifikasi data Anda sekarang di link berikut.' "
            "'Dapatkan diskon 90% semua produk, klik link ini sebelum kehabisan.' "
            "Semua pola pesan di atas teridentifikasi sebagai spam berbahaya."
        )
    },
    {
        "chunk_id": "chunk_E_002",
        "category": "sms_spam_example",
        "modus_type": "social_engineering",
        "risk_level": "tinggi",
        "source": "Indonesia SMS Spam Dataset",
        "source_url": "https://github.com/bopbi/indonesia-sms-spam-dataset",
        "document_type": "dataset_pesan",
        "author": "bopbi",
        "language": "Indonesian",
        "content": (
            "Contoh SMS spam nyata kategori social engineering dari dataset Indonesia: "
            "'Halo, saya dari CS BCA. Ada transaksi mencurigakan di akun Anda. Berikan kode OTP.' "
            "'Ini pihak Tokopedia, akun Anda bermasalah. Segera berikan PIN untuk verifikasi.' "
            "'Dari tim keamanan Shopee: konfirmasi identitas Anda dengan kirim foto KTP dan selfie.' "
            "Ciri khas: menggunakan nama lembaga resmi, menciptakan urgensi, meminta data sensitif "
            "melalui saluran tidak resmi. Semua pesan ini adalah penipuan."
        )
    },
]

print(f"✅ Knowledge base dimuat: {len(KNOWLEDGE_BASE)} chunk")
print("\n📋 Daftar chunk:")
for chunk in KNOWLEDGE_BASE:
    print(f"  [{chunk['chunk_id']}] {chunk['category']} | {chunk['modus_type']} | Sumber: {chunk['author']}")

## 🔢 Sel 4 — Build Vector Database (Embedding + FAISS)

Setiap chunk di-embed menggunakan `sentence-transformers` lalu disimpan di FAISS untuk semantic search.

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("⏳ Memuat embedding model (pertama kali bisa 1-2 menit)...")
# Model multilingual yang mendukung Bahasa Indonesia
EMBED_MODEL = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Embedding model siap")

# Embed semua chunk
print("\n⏳ Membuat embedding untuk semua chunk...")
corpus = [chunk["content"] for chunk in KNOWLEDGE_BASE]
embeddings = EMBED_MODEL.encode(corpus, show_progress_bar=True, normalize_embeddings=True)

# Build FAISS index (Inner Product = Cosine Similarity karena sudah normalize)
DIM = embeddings.shape[1]
INDEX = faiss.IndexFlatIP(DIM)
INDEX.add(np.array(embeddings, dtype="float32"))

print(f"\n✅ FAISS index dibangun")
print(f"   Dimensi embedding : {DIM}")
print(f"   Jumlah vektor     : {INDEX.ntotal}")

## 🔍 Sel 5 — Fungsi Retrieval

Fungsi `retrieve_chunks()` menerima query teks, mengubahnya menjadi vektor, lalu mencari chunk paling relevan menggunakan cosine similarity di FAISS.

In [ ]:
def retrieve_chunks(query: str, top_k: int = 3, min_score: float = 0.2):
    """
    Semantic search menggunakan FAISS.
    
    Args:
        query     : teks query dari user
        top_k     : jumlah chunk yang diambil
        min_score : threshold minimum similarity score
    
    Returns:
        List of dict (chunk + similarity_score)
    """
    query_vec = EMBED_MODEL.encode([query], normalize_embeddings=True)
    scores, indices = INDEX.search(np.array(query_vec, dtype="float32"), top_k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        if float(score) < min_score:
            continue
        chunk = dict(KNOWLEDGE_BASE[idx])  # copy agar tidak mutate original
        chunk["similarity_score"] = round(float(score), 4)
        results.append(chunk)
    
    return results


# ── Uji coba retrieval ───────────────────────────────────────────────
test_query = "Dapat SMS menang hadiah Rp50 juta dari BRI, diminta klik link"
retrieved = retrieve_chunks(test_query)

print(f"🔍 Query  : {test_query}")
print(f"📦 Chunk ditemukan: {len(retrieved)}\n")
for r in retrieved:
    print(f"  [{r['chunk_id']}] similarity={r['similarity_score']}")
    print(f"  Kategori : {r['category']} | Modus: {r['modus_type']}")
    print(f"  Sumber   : {r['source']}")
    print(f"  Preview  : {r['content'][:100]}...")
    print()

## 🤖 Sel 6 — LLM Generation dengan Guardrails

Fungsi utama RAG: menggabungkan retrieved context dengan prompt, lalu memanggil Claude API.  
System prompt mengandung **guardrails** sesuai dokumen UTS.

In [ ]:
import anthropic
import json
import textwrap

CLIENT = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# ── System Prompt (Guardrails) ───────────────────────────────────────
SYSTEM_PROMPT = textwrap.dedent("""
    Kamu adalah SafeCheck AI, asisten keamanan digital berbahasa Indonesia yang membantu
    pengguna mendeteksi penipuan digital seperti phishing, scam, dan social engineering.

    GUARDRAILS — WAJIB DIPATUHI:
    1. Berbasis dokumen: HANYA jawab berdasarkan dokumen yang tersedia dalam konteks.
       Jangan mengarang informasi di luar dokumen.
    2. Wajib ada source: Setiap jawaban harus menyebutkan sumber dokumen.
    3. Akui ketidaktahuan: Jika tidak ditemukan di dokumen, nyatakan
       "Tidak ditemukan dalam dokumen yang tersedia."
    4. Domain restriction: HANYA jawab terkait keamanan digital dan penipuan online.
       Tolak pertanyaan di luar topik ini dengan sopan.
    5. Jika risiko tinggi: Selalu tampilkan kontak resmi (OJK 157, kontak bank).

    FORMAT JAWABAN — Selalu gunakan struktur berikut:
    🔴/🟡/🟢 STATUS RISIKO: [Tinggi/Sedang/Rendah]
    📋 ANALISIS: [penjelasan singkat berdasarkan dokumen]
    ⚠️  INDIKATOR BAHAYA: [daftar poin]
    ✅ REKOMENDASI: [tindakan yang harus dilakukan]
    📚 SOURCE: [nama dokumen sumber]
    📞 KONTAK RESMI: [jika risiko tinggi]
""").strip()


def build_rag_prompt(user_query: str, retrieved_chunks: list) -> str:
    """
    Susun prompt RAG: gabungkan query + retrieved context.
    """
    if not retrieved_chunks:
        context_text = "Tidak ditemukan dokumen relevan dalam knowledge base."
    else:
        context_parts = []
        for i, chunk in enumerate(retrieved_chunks, 1):
            context_parts.append(
                f"[Dokumen {i}]\n"
                f"Chunk ID    : {chunk['chunk_id']}\n"
                f"Kategori    : {chunk['category']}\n"
                f"Modus       : {chunk['modus_type']}\n"
                f"Risiko      : {chunk['risk_level']}\n"
                f"Sumber      : {chunk['source']}\n"
                f"Similarity  : {chunk['similarity_score']}\n"
                f"Isi         : {chunk['content']}"
            )
        context_text = "\n\n".join(context_parts)
    
    prompt = f"""
DOKUMEN REFERENSI YANG TERSEDIA:
{context_text}

PESAN YANG DICEK PENGGUNA:
\"{user_query}\"

Berdasarkan dokumen referensi di atas, analisis apakah pesan ini mengandung indikasi
penipuan digital. Gunakan format yang sudah ditentukan.
"""
    return prompt.strip()


def safecheck_analyze(user_query: str, top_k: int = 3, verbose: bool = True):
    """
    Pipeline RAG lengkap: retrieve → augment → generate.
    
    Args:
        user_query : teks pesan yang ingin dicek
        top_k      : jumlah chunk yang diambil
        verbose    : tampilkan detail pipeline
    
    Returns:
        dict berisi: query, retrieved_chunks, answer, sources
    """
    if verbose:
        print("=" * 60)
        print("🛡️  SafeCheck AI — RAG Pipeline")
        print("=" * 60)
        print(f"\n📨 Query: {user_query}")
        print("\n[Step 1] 🔢 Embedding query...")
    
    # ── Step 1 & 2: Retrieve ─────────────────────────────────────────
    retrieved = retrieve_chunks(user_query, top_k=top_k)
    
    if verbose:
        print(f"[Step 2] 🔍 Vector search selesai — {len(retrieved)} chunk ditemukan")
        for r in retrieved:
            print(f"         → [{r['chunk_id']}] {r['category']} | score={r['similarity_score']}")
    
    # ── Step 3: Augment ──────────────────────────────────────────────
    rag_prompt = build_rag_prompt(user_query, retrieved)
    
    if verbose:
        print("[Step 3] 📄 Context disiapkan untuk LLM")
        print("[Step 4] 🤖 Menghubungi Claude API...")
    
    # ── Step 4: Generate ─────────────────────────────────────────────
    response = CLIENT.messages.create(
        model="claude-haiku-4-5-20251001",  # Model ringan, cocok untuk demo
        max_tokens=1000,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": rag_prompt}]
    )
    
    answer = response.content[0].text
    sources = list({r["source"] for r in retrieved})
    
    if verbose:
        print("\n" + "─" * 60)
        print("📋 HASIL ANALISIS SafeCheck AI:")
        print("─" * 60)
        print(answer)
        print("─" * 60)
        print(f"\n📚 Sumber dokumen: {', '.join(sources)}")
        print(f"🔢 Token digunakan: {response.usage.input_tokens} in / {response.usage.output_tokens} out")
    
    return {
        "query": user_query,
        "retrieved_chunks": retrieved,
        "answer": answer,
        "sources": sources,
        "token_usage": {
            "input": response.usage.input_tokens,
            "output": response.usage.output_tokens
        }
    }

print("✅ Fungsi RAG siap digunakan")

## 🧪 Sel 7 — Uji 10 Pertanyaan User

Menguji semua 10 pertanyaan dari dokumen UTS satu per satu.

In [ ]:
# 10 Pertanyaan user dari dokumen UTS
USER_QUESTIONS = [
    "Saya dapat SMS katanya menang hadiah dari BRI, beneran gak?",
    "Ada yang minta OTP saya lewat WhatsApp ngaku dari Tokopedia, aman?",
    "Dapat email dari BCA minta verifikasi akun, link-nya harus diklik?",
    "Link bit.ly dikirim orang asing di WhatsApp, bahaya gak kalau dibuka?",
    "Nomor ini ngaku CS Shopee tapi bukan dari aplikasi resmi, gimana?",
    "Diminta transfer dulu buat cairkan hadiah, ini penipuan?",
    "SMS bilang paket tertahan di bea cukai dan harus bayar, valid?",
    "Ada lowongan kerja minta foto KTP dan selfie sebelum interview, normal?",
    "Dapat pesan akun akan diblokir dalam 24 jam, perlu direspons?",
    "WhatsApp blast dari nomor tidak dikenal kirim link promo diskon 90%, aman?"
]

# Jalankan satu pertanyaan dulu sebagai demo utama
# Ganti index [0] untuk mencoba pertanyaan lain
result = safecheck_analyze(USER_QUESTIONS[0])

In [ ]:
# Jalankan sel ini untuk mencoba pertanyaan lain
# Ganti nomor index (0-9) sesuai pertanyaan yang ingin dicoba

QUESTION_INDEX = 1  # <- Ganti di sini (0 sampai 9)

print(f"Pertanyaan #{QUESTION_INDEX + 1}: {USER_QUESTIONS[QUESTION_INDEX]}\n")
result2 = safecheck_analyze(USER_QUESTIONS[QUESTION_INDEX])

## 📊 Sel 8 — Evaluasi Sederhana (3 Pertanyaan Uji)

Evaluasi minimal sesuai dokumen UTS: cek relevansi retrieval, akurasi jawaban, dan ketersediaan source.

In [ ]:
import pandas as pd

# 3 Pertanyaan evaluasi dari dokumen UTS
EVAL_QUERIES = [
    {
        "query": "Dapat SMS menang hadiah Rp50 juta dari bank, diminta klik link.",
        "expected_category": "scam_hadiah",
        "expected_risk": "tinggi"
    },
    {
        "query": "Ada yang telepon minta OTP saya, ngaku dari CS bank.",
        "expected_category": "social_engineering",
        "expected_risk": "tinggi"
    },
    {
        "query": "Dapat email domain mirip Tokopedia tapi ada huruf tambahan.",
        "expected_category": "phishing_email",
        "expected_risk": "tinggi"
    }
]

eval_results = []

for i, eval_item in enumerate(EVAL_QUERIES, 1):
    print(f"\n{'='*60}")
    print(f"EVALUASI #{i}: {eval_item['query']}")
    print(f"{'='*60}")
    
    result = safecheck_analyze(eval_item["query"], verbose=True)
    
    # Evaluasi retrieval: apakah kategori yang diharapkan ada di chunk yang diambil?
    retrieved_categories = [r["category"] for r in result["retrieved_chunks"]]
    retrieval_relevant = eval_item["expected_category"] in retrieved_categories
    
    # Evaluasi source: apakah ada sumber yang dikutip?
    has_source = len(result["sources"]) > 0
    
    # Evaluasi risk level: apakah keyword risiko tinggi ada di jawaban?
    answer_lower = result["answer"].lower()
    risk_correct = "tinggi" in answer_lower or "berbahaya" in answer_lower or "waspada" in answer_lower
    
    eval_results.append({
        "No": i,
        "Query": eval_item["query"][:50] + "...",
        "Retrieval Relevan": "✅" if retrieval_relevant else "❌",
        "Source Tersedia": "✅" if has_source else "❌",
        "Risk Level Benar": "✅" if risk_correct else "❌",
        "Chunk Ditemukan": len(result["retrieved_chunks"]),
        "Kategori Retrieved": ", ".join(retrieved_categories)
    })

# Tampilkan tabel evaluasi
print("\n" + "=" * 60)
print("📊 RINGKASAN EVALUASI")
print("=" * 60)
df_eval = pd.DataFrame(eval_results)
print(df_eval.to_string(index=False))

# Hitung skor
total = len(eval_results)
r_score = sum(1 for r in eval_results if r["Retrieval Relevan"] == "✅")
s_score = sum(1 for r in eval_results if r["Source Tersedia"] == "✅")
k_score = sum(1 for r in eval_results if r["Risk Level Benar"] == "✅")

print(f"\n📈 Skor Evaluasi:")
print(f"   Retrieval Relevan : {r_score}/{total} ({r_score/total*100:.0f}%)")
print(f"   Source Tersedia   : {s_score}/{total} ({s_score/total*100:.0f}%)")
print(f"   Risk Level Benar  : {k_score}/{total} ({k_score/total*100:.0f}%)")

## 🔄 Sel 9 — Mode Interaktif

Coba sendiri! Masukkan pesan mencurigakan dan SafeCheck AI akan menganalisisnya.

In [ ]:
# ── Mode Interaktif ──────────────────────────────────────────────────
# Ubah teks di bawah dengan pesan yang ingin kamu cek

PESAN_KAMU = """
Selamat! Anda terpilih sebagai pemenang undian berhadiah dari Bank Mandiri.
Total hadiah Rp 75.000.000. Untuk klaim, klik link: bit.ly/hadiahMandiri2024
dalam 24 jam. Hubungi admin: 0812-XXXX-XXXX
"""

result_interaktif = safecheck_analyze(PESAN_KAMU.strip())

## 🔬 Sel 10 — Uji Guardrail: Out-of-Domain

Membuktikan bahwa guardrail bekerja — AI menolak pertanyaan di luar topik keamanan digital.

In [ ]:
# Test guardrail: pertanyaan di luar domain
OUT_OF_DOMAIN_QUERIES = [
    "Apa ibu kota Australia?",
    "Tolong buatkan resep nasi goreng",
    "Siapa presiden Indonesia?"
]

print("🛡️ UJI GUARDRAIL — Pertanyaan di Luar Domain\n")
for q in OUT_OF_DOMAIN_QUERIES:
    print(f"Query: {q}")
    result_ood = safecheck_analyze(q, verbose=False)
    print(f"Jawaban: {result_ood['answer'][:200]}...")
    print("─" * 50)

## 📊 Sel 11 — Visualisasi Similarity Score

Visualisasi skor similarity antara query dan setiap chunk di knowledge base.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def visualize_retrieval(query: str, top_k: int = None):
    """
    Visualisasi similarity score semua chunk untuk satu query.
    """
    if top_k is None:
        top_k = len(KNOWLEDGE_BASE)
    
    query_vec = EMBED_MODEL.encode([query], normalize_embeddings=True)
    scores, indices = INDEX.search(
        np.array(query_vec, dtype="float32"), len(KNOWLEDGE_BASE)
    )
    
    chunk_ids = [KNOWLEDGE_BASE[i]["chunk_id"] for i in indices[0]]
    categories = [KNOWLEDGE_BASE[i]["category"] for i in indices[0]]
    score_vals = [float(s) for s in scores[0]]
    
    # Warna per kategori
    cat_colors = {
        "scam_hadiah": "#D85A30",
        "social_engineering": "#534AB7",
        "phishing_email": "#185FA5",
        "data_harvesting": "#BA7517",
        "sms_spam_example": "#639922"
    }
    colors = [cat_colors.get(c, "#888780") for c in categories]
    
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(chunk_ids, score_vals, color=colors, height=0.6)
    
    # Garis threshold
    ax.axvline(x=0.2, color="red", linestyle="--", linewidth=1, alpha=0.7, label="Min threshold (0.2)")
    
    # Label nilai
    for bar, val in zip(bars, score_vals):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                f"{val:.3f}", va="center", fontsize=9)
    
    ax.set_xlabel("Cosine Similarity Score")
    ax.set_title(f"SafeCheck AI — Retrieval Score\nQuery: \"{query[:60]}...\"" if len(query) > 60 else f"SafeCheck AI — Retrieval Score\nQuery: \"{query}\"")
    ax.set_xlim(0, max(score_vals) + 0.1)
    
    # Legend kategori
    patches = [mpatches.Patch(color=v, label=k) for k, v in cat_colors.items()]
    ax.legend(handles=patches, loc="lower right", fontsize=8)
    
    plt.tight_layout()
    plt.savefig("retrieval_scores.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("💾 Gambar disimpan: retrieval_scores.png")


# Visualisasi untuk 3 query evaluasi
for eq in EVAL_QUERIES:
    visualize_retrieval(eq["query"])

---

## ✅ Ringkasan Sistem SafeCheck AI

| Komponen | Detail |
|---|---|
| **Nama AI** | SafeCheck AI |
| **Arsitektur** | RAG (Retrieval-Augmented Generation) |
| **Embedding Model** | paraphrase-multilingual-MiniLM-L12-v2 |
| **Vector Store** | FAISS (IndexFlatIP / Cosine Similarity) |
| **LLM** | Claude (Anthropic) via API |
| **Dokumen Sumber** | BSSN 2024, OJK Anti Fraud 2024, SMS Spam Dataset |
| **Chunk Size** | 300–500 token per chunk |
| **Overlap** | 50 token |
| **Metadata** | chunk_id, category, modus_type, risk_level, source, author |
| **Guardrails** | Berbasis dokumen, wajib source, domain restriction, kontak resmi |
| **Evaluasi** | Retrieval relevance, source availability, risk level accuracy |

**Referensi Dokumen:**
- BSSN: https://alika.pesisirbaratkab.go.id/ebook/lanskap-keamanan-siber-indonesia-2024
- OJK: https://ojk.go.id/id/Publikasi/Roadmap-dan-Pedoman/ITSK/Documents/PANDUAN%20STRATEGI%20ANTI%20FRAUD%20(ITSK)%202024.pdf
- Dataset: https://github.com/bopbi/indonesia-sms-spam-dataset